# 0.加载环境变量

In [ ]:
from dotenv import load_dotenv
from langchain_classic.chains.question_answering.map_reduce_prompt import messages

load_dotenv()

# 1.初始化并调用模型

langchain 提供了俩种常见函数用来初始化模型：
- 使用`init_chat_model`函数，由LangChain自动创建模型对象.
- 使用不通模型对应的类，手动创建模型对象

## 1.1`init_chat_model`

官方最推荐的方式是使用`init_chat_model`函数

### 基于名称推断模型提供商
使用`init_chat_model`函数，需要从langChain支持的模型提供者（Model Provider）中选择一个模型。而LangChain根据模型名称自动初始化与模型的连接

LangChain支持的模型列表参考官网链接：
https://docs.langchain.com/oss/python/integrations/providers/overview

步骤：
- 安装模型依赖：`uv add langchain langchain-deepseek`
- 在 .env 中配置模型的 api_key
- 调用`init_chat_model`函数，传入正确的模型名称


In [ ]:
# 导入 LangChain 的初始化模型的函数
from langchain.chat_models import init_chat_model

# 在 .env 中配置 DEEPSEEK_API_KEY=sk-31bbc60d5513437c9f99c2dfbbc1f883

# 调用 init_chat_model 函数初始化模型
# 参数 model 用来指定模型名称，langChain 会根据模型名字自动设定 base_url, 并从环境变量中获取 api_key
model = init_chat_model(model = "deepseek-chat")

In [ ]:
print(type(model))

### 自定义模型提供商
`init_chat_model`默认会根据模型名称自动确定模型的提供者的`base_url`，并从环境变量中获取 api_key，但前提是必须是langchain支持的模型平台：

The following providers will be inferred based on these model prefixes:

            - `gpt-...` | `o1...` | `o3...`       -> `openai`
            - `claude...`                         -> `anthropic`
            - `amazon...`                         -> `bedrock`
            - `gemini...`                         -> `google_vertexai`
            - `command...`                        -> `cohere`
            - `accounts/fireworks...`             -> `fireworks`
            - `mistral...`                        -> `mistralai`
            - `deepseek...`                       -> `deepseek`
            - `grok...`                           -> `xai`
            - `sonar...`                          -> `perplexity`
            - `solar...`                          -> `upstage`

对于其他模型，我们必须自定义模型参数来访问：

例如，阿里云百炼的 qwen-max，
- 我们需要在环境变量中定义`api_key`和`base_url`
- 然后在`init_chat_model`中指定`model`、`model_provider`、`base_url`和`api_key`

In [ ]:
# 接收加载环境变量中的 base_url 和 qpi_key
import os

base_url = os.getenv("DASHSCOPE_BASE_URL")
api_key = os.getenv("DASHSCOPE_API_KEY")

model = init_chat_model(
    model = "qwen-max",
    model_provider="openai",
    base_url = base_url,
    api_key = api_key
)

In [ ]:
print(type(model))

### 调整模型参数
除了修改模型提供者以外，`init_chat_model`函数还支持传入其他参数来调整模型的行为，例如:
- `temperature`: 控制生成文本的随机性，值越小越确定，值越大越随机
- `max_tokens`: 控制生成文本的最大长度
- `top_p`: 控制生成文本的多样性，值越小越保守，值越大越开放
- `timeout`: 控制模型响应的最长时间，单位为秒
- `max_retries`: 控制模型请求的最大重试次数

In [ ]:
model = init_chat_model(
    model = "qwen-max",
    model_provider="openai",
    base_url = base_url,
    api_key = api_key,
    temperature = 1.5,
    top_p = 0.9,
)

print(type(model))

## 1.2 使用 model 类
`init_chat_model`函数底层就是帮我们利用 Model 类创建对象，但只支持有限的模型

而在LangChain的社区，除了LangChain官方提供的 Model，还有些类是社区提供的，更丰富多样

具体支持的模型，可以查看官网地址：https://docs.langchain.com/oss/python/integrations/chat

例如，我们使用社区版本的 Model 类来访问阿里云百炼的通义千问模型：
1. 首先，安装依赖LangChain社区依赖：

   `uv add langchain-community`

   阿里云百炼依赖：

   `uv add dashscope`

2. 在 .env 中配置模型的 api_key 和 base_url

3. 使用Model类初始化模型

In [ ]:
from langchain_community.chat_models.tongyi import ChatTongyi

# 使用 Model 类初始化模型
model = ChatTongyi(
    model = "qwen-max",

)

In [ ]:
print(type(model))

# 2.访问模型

LangChain提供了俩个不同的函数来访问模型：
- `invoke`: 阻塞时访问
- `stream`: 流式访问

## 2.1 invoke
`invoke`函数会等待模型生成完整的响应后才返回结果，适合需要一次性获取完整答案的场景

In [ ]:
# 通过 invoke 函数访问模型，需要阻塞等待模型生成结果
response = model.invoke("你是谁？")

# 查看响应内容
print(response)

In [ ]:
# 调用 invoke 函数，传入消息数组
response = model.invoke([
    {"role": "system", "content": "你是我的人工智能助手，协助我解答问题"},
    {"role": "user", "content": "你是谁？"}
])
print(response.content)

## 2.2 stream
`stream`函数会在模型生成响应的过程中逐步返回结果，适合需要实时获取模型输出的场景，例如聊天应用或者需要展示生成过程的场景

In [ ]:
# 通过 stream 函数访问模型
stream = model.stream("你是谁？")
print(type(stream))

In [ ]:
for chunk in stream:
    print(chunk)


# 3.在智能体中使用模型

## 3.1 在智能体中使用模型
LangChain 提供了一个`create_agent`函数用来快速创建智能体，调用`create_agent`时，需要指定一个模型，
- 使用初始化好的模型对象
- 使用模型名称，让LangChain自动化初始化模型

In [ ]:
from langchain.agents import create_agent

# 1.使用初始化好的 model 创建 Agent
agent = create_agent(model=model)

In [ ]:
# 2.指定 Model 名称，由 LanfChain 自动初始化模型
agent = create_agent(model="deepseek-chat")
print(type(agent))

## 3.2 调用智能体

智能体调用与模型调用类似，也支持俩种方式：

- `invoke`: 阻塞时访问智能体
- `stream`: 流式访问智能体

需要注意的是，智能体调用时需要传入一个`dict`，其中必须包含一个`messages`字段，也就是消息的列表。

### 阻塞式调用


In [ ]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "你是谁？"}]
})

print(response)

### 流式调用

In [ ]:
messages = agent.stream(
    {"messages": [{"role": "user", "content": "你是谁？"}]},
    stream_mode="messages"
)

print(type(messages))

In [ ]:
# 遍历 stream 结果，实时打印 AI 的回复
for token, metadata in messages:
    if token.content:
        print(token.content, end="", flush=True)